In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import paths


# Gridcell Surrogate KFolds
Loads the `CV_summary_*.json` saved by `GP_gridcell_kfolds_masked_10seeds_rawr2.py`.

Aggregation strategy: average folds within each seed → per-seed mean; then mean ± std across seeds.

In [1]:
import glob
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Configuration
Set `save_dir` to the directory where `surrogate_kfolds_masked.py` wrote its output. The most recent `CV_summary_*.json` will be loaded automatically.

In [2]:
save_dir = str(paths.CV_RESULTS_DIR / "10seed_GP_gridcell_raw") + "/"

# Auto-find most recent CV summary JSON, or override manually
matches  = sorted(glob.glob(os.path.join(save_dir, 'CV_summary_*.json')))
cv_file  = matches[-1] if matches else None
# cv_file = os.path.join(save_dir, 'CV_summary_TIMESTAMP.json')

print(f'Loading: {cv_file}')
with open(cv_file) as f:
    cv_summary = json.load(f)

# Flatten {model: [fold_dict, ...]} into a single DataFrame
records = []
for model, fold_list in cv_summary.items():
    records.extend(fold_list)
df = pd.DataFrame(records)

print(f'Loaded {len(df)} records')
print(f'Models : {sorted(df["model"].unique())}')
if 'seed' in df.columns:
    print(f'Seeds  : {sorted(df["seed"].unique())}')
print(f'Folds  : {sorted(df["fold"].unique())}')
df.head()

Loading: /global/cfs/cdirs/e3sm/jpaige3/ESEm/CV_Saved_Model_Data_masked/10seeds_GP_gridcell_raw/CV_summary_gridcell_2026-05-21_11-06-20.json
Loaded 50 records
Models : ['GP']
Seeds  : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Folds  : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


,model,seed,fold,PCP_train_r2_norm,PCP_train_rmse_norm,PCP_train_r2_phys,PCP_train_rmse_phys,PCP_train_r2_norm_raw,PCP_train_r2_phys_raw,PCP_test_r2_norm,...,OLR_train_r2_phys,OLR_train_rmse_phys,OLR_train_r2_norm_raw,OLR_train_r2_phys_raw,OLR_test_r2_norm,OLR_test_rmse_norm,OLR_test_r2_phys,OLR_test_rmse_phys,OLR_test_r2_norm_raw,OLR_test_r2_phys_raw
0,GP,0,0,0.999877,0.010826,0.999956,2.280831e-11,"[0.9999420383900599, 0.9999407509937831, 0.999...","[0.9999580288930371, 0.999977260903011, 0.9999...",-3.933241e+06,...,0.906000,1.970042,"[0.9499564944962434, 0.9441800115751352, 0.957...","[0.882738193619139, 0.9095020411073577, 0.9099...",-0.004886,0.492503,0.775823,2.177223,"[0.7999138790334377, 0.7265453972026026, 0.854...","[0.8743282547450726, 0.823951166382695, 0.8020..."
1,GP,0,1,0.999432,0.023223,0.999802,4.214745e-11,"[0.9996828949277692, 0.9997511875011535, 0.999...","[0.9997767189792308, 0.9999164590480203, 0.999...",-3.637974e+01,...,0.945495,1.536657,"[0.9569020287432265, 0.947084011091678, 0.9633...","[0.9326239847007229, 0.9516277584208161, 0.950...",-0.863114,0.756180,0.058487,3.622112,"[0.8045631477660999, 0.814436381982972, 0.6952...","[0.23915422319531365, 0.36646608109499257, 0.3..."
2,GP,0,2,0.990128,0.096651,0.996295,2.045003e-10,"[0.9950817384746228, 0.995422828358432, 0.9965...","[0.9956291308172831, 0.9979673659942955, 0.998...",-2.143430e+08,...,0.996081,0.379719,"[0.9958966701879424, 0.9947795598091324, 0.997...","[0.9951498944791343, 0.9959257076935316, 0.995...",-1.021398,0.767517,0.252022,4.537552,"[0.7956179999702112, 0.7346306203323896, 0.743...","[0.1734635715330225, 0.3857467537364915, 0.380..."
3,GP,0,3,0.951745,0.213015,0.982848,4.081985e-10,"[0.9739063977672889, 0.9730154760811348, 0.977...","[0.9831930085563955, 0.9912915350161305, 0.992...",-5.404337e+00,...,0.995495,0.325915,"[0.9954361554764859, 0.9942242132148114, 0.996...","[0.9921974665116472, 0.9951354320379162, 0.995...",0.386873,1.382272,0.616732,6.137003,"[0.8687696195421697, 0.8481142723524662, 0.870...","[0.7204261633105284, 0.7456593236952302, 0.714..."
4,GP,0,4,0.901222,0.307292,0.963067,6.106213e-10,"[0.9658245409466162, 0.9338922802873346, 0.953...","[0.967494031606881, 0.9794553584267648, 0.9822...",-5.866886e+08,...,0.979880,0.900780,"[0.986168028221247, 0.9827335474185388, 0.9849...","[0.9771707573771153, 0.9811261842705276, 0.981...",-2.789674,0.835809,0.074344,4.853557,"[0.7406374673583249, 0.7658025715579898, 0.818...","[-0.07546219739372195, 0.38102125942308607, 0...."


## Aggregate: mean ± std across seeds
Step 1 — for each (model, seed), average the metric across its 5 folds.  
Step 2 — across the resulting per-seed means, compute mean and std.

In [3]:
VAR_NAMES  = ['PCP', 'TLWP', 'OSR', 'OLR']
MODEL_ORDER = ['GP', 'CNN', 'RF', 'MLR', 'Spline']
models_present = [m for m in MODEL_ORDER if m in df['model'].unique()]

group_cols = ['model', 'seed'] if 'seed' in df.columns else ['model']

# Exclude list-valued columns (raw per-gridpoint R² arrays) — keep only scalar metrics
metric_cols = [
    c for c in df.columns
    if c not in ('model', 'seed', 'fold')
    and not c.endswith('_raw')
    and df[c].apply(lambda x: np.isscalar(x)).all()
]

# Step 1: mean across folds within each (model, seed)
seed_means = df.groupby(group_cols)[metric_cols].mean().reset_index()

# Step 2: mean and std across seeds for each model
agg_mean = seed_means.groupby('model')[metric_cols].mean()
agg_std  = seed_means.groupby('model')[metric_cols].std(ddof=1)

print('=== Mean across seeds (test R², physical space) ===')
test_phys_cols = [c for c in metric_cols if 'test_r2_phys' in c]
display_mean = agg_mean[test_phys_cols].copy()
display_mean['Mean'] = display_mean.mean(axis=1)
display(display_mean.round(3))

print('\n=== Std across seeds ===')
display_std = agg_std[test_phys_cols].copy()
display_std['Mean'] = display_std.mean(axis=1)
display(display_std.round(3))

=== Mean across seeds (test R², physical space) ===


,PCP_test_r2_phys,TLWP_test_r2_phys,OSR_test_r2_phys,OLR_test_r2_phys,Mean
model,,,,,
GP,0.611,0.76,0.671,0.292,0.583



=== Std across seeds ===


,PCP_test_r2_phys,TLWP_test_r2_phys,OSR_test_r2_phys,OLR_test_r2_phys,Mean
model,,,,,
GP,0.029,0.016,0.027,0.125,0.049


In [5]:
print('=== Mean across seeds (test RMSE, physical space) ===')
test_phys_cols = [c for c in metric_cols if 'test_rmse_phys' in c]
display_mean = agg_mean[test_phys_cols].copy()
display(display_mean)

print('\n=== Std across seeds ===')
display_std = agg_std[test_phys_cols].copy()
display(display_std)

=== Mean across seeds (test RMSE, physical space) ===


,PCP_test_rmse_phys,TLWP_test_rmse_phys,OSR_test_rmse_phys,OLR_test_rmse_phys
model,,,,
GP,1.784071e-09,0.009794,5.113535,4.10212



=== Std across seeds ===


,PCP_test_rmse_phys,TLWP_test_rmse_phys,OSR_test_rmse_phys,OLR_test_rmse_phys
model,,,,
GP,8.547356e-11,0.000289,0.150309,0.242382
